# Week 6 – Feature Engineering and Market Metrics

**Objective:** Use the cleaned MLS sold dataset to engineer market metrics that will support later Tableau dashboards.

This notebook creates:

- Price Ratio
- Close-to-Original-List Ratio
- Price Per Sq Ft
- Days on Market
- Year / Month / YrMo
- Listing-to-Contract Days
- Contract-to-Close Days
- Sample output table
- Segmented summary tables by county, property type, MLS area, and office


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np


In [7]:
# Base project directory
BASE_DIR = Path("/Users/amyliu/Desktop/IDX")

# Input file: change this if your file name is different
SOLD_FILE = BASE_DIR / "data" / "generated" / "sold_with_rates_week4-5.csv"
LISTINGS_FILE = BASE_DIR / "data" / "generated" / "listing_with_rates_week4-5.csv"

# Output directory
OUTPUT_DIR = BASE_DIR / "data" / "generated" / "week6"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Sold input file:", SOLD_FILE)
print("Listings input file:", LISTINGS_FILE)
print("Output directory:", OUTPUT_DIR)

if not SOLD_FILE.exists():
    raise FileNotFoundError(f"Sold file not found: {SOLD_FILE}")

if not LISTINGS_FILE.exists():
    raise FileNotFoundError(f"Listings file not found: {LISTINGS_FILE}")

Sold input file: /Users/amyliu/Desktop/IDX/data/generated/sold_with_rates_week4-5.csv
Listings input file: /Users/amyliu/Desktop/IDX/data/generated/listing_with_rates_week4-5.csv
Output directory: /Users/amyliu/Desktop/IDX/data/generated/week6


In [8]:
sold = pd.read_csv(SOLD_FILE, low_memory=False)

print(f"Rows loaded: {len(sold):,}")
print(f"Columns loaded: {sold.shape[1]:,}")

sold.head()

Rows loaded: 591,733
Columns loaded: 84


,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,PoolPrivateYN,OriginalListPrice,ListingKey,CloseDate,ClosePrice,ListAgentFirstName,...,invalid_dom_flag,invalid_bedrooms_flag,invalid_bathrooms_flag,listing_after_close_flag,purchase_after_close_flag,negative_timeline_flag,missing_coord_flag,zero_coord_flag,positive_longitude_flag,implausible_coord_flag
0,Mlslistings,Mlslistings,"Carpet,Tile,Wood",True,False,499000.0,551985747,2024-01-26,240000.0,Joan,...,False,False,False,False,False,False,True,False,False,False
1,HighDesert,HighDesert,NaN,NaN,NaN,0.0,535486633,2024-01-24,950.0,Elizabeth,...,False,False,False,False,False,False,False,False,False,False
2,OrangeCounty,OrangeCounty,NaN,True,NaN,75000.0,529986282,2024-01-16,45000.0,Joseph,...,False,False,False,False,False,False,False,False,False,False
3,InlandValleys,InlandValleys,NaN,True,NaN,199000.0,529618166,2024-01-08,141500.0,CAROL,...,False,False,False,False,False,False,False,False,False,False
4,SouthwestRiversideCounty,SouthwestRiversideCounty,NaN,True,NaN,19500.0,522614340,2024-01-17,15000.0,Jeremie,...,False,False,False,False,False,False,False,False,False,False


In [23]:
listings = pd.read_csv(LISTINGS_FILE, low_memory=False)

print(f"Rows loaded: {len(listings):,}")
print(f"Columns loaded: {listings.shape[1]:,}")

listings.head()

Rows loaded: 852,963
Columns loaded: 74


,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,...,invalid_dom_flag,invalid_bedrooms_flag,invalid_bathrooms_flag,listing_after_close_flag,purchase_after_close_flag,negative_timeline_flag,missing_coord_flag,zero_coord_flag,positive_longitude_flag,implausible_coord_flag
0,90000.0,1075010398,miriamlara03@gmail.com,NaN,NaN,Miriam,Lara,34.097939,-117.909653,1045 N Azusa 61,...,False,False,False,False,False,False,False,False,False,False
1,1500000.0,1074974457,janelle@judsonre.com,NaN,NaN,Janelle,Judson,33.121241,-117.081614,NaN,...,False,False,False,False,False,False,False,False,False,False
2,1340000.0,1074973329,haleh360@Gmail.com,NaN,NaN,Haleh,Dowlatshahi,34.052207,-118.408445,2220 Avenue Of The Stars 2704,...,False,False,False,False,False,False,False,False,False,False
3,2500000.0,1074954552,Reneechen@yourhomesoldguaranteed.com,NaN,NaN,Renee,Chen,33.496363,-117.691677,16 Palisades,...,False,False,False,False,False,False,False,False,False,False
4,3150000.0,1074936537,anader@dppre.com,NaN,NaN,Margaret,Nader,34.119345,-118.111254,1615 Waverly Road,...,False,False,False,False,False,False,False,False,False,False


## 3. Inspect available columns

Before creating features, check whether the required source columns are available.

In [10]:
required_cols = [
    "CloseDate",
    "PurchaseContractDate",
    "ListingContractDate",
    "ClosePrice",
    "OriginalListPrice",
    "LivingArea",
    "DaysOnMarket",
    "PropertyType",
    "PropertySubType",
    "CountyOrParish",
    "MLSAreaMajor",
    "ListOfficeName",
    "BuyerOfficeName"
]

available_check = pd.DataFrame({
    "column": required_cols,
    "exists": [col in sold.columns for col in required_cols]
})

available_check

,column,exists
0,CloseDate,True
1,PurchaseContractDate,True
2,ListingContractDate,True
3,ClosePrice,True
4,OriginalListPrice,True
5,LivingArea,True
6,DaysOnMarket,True
7,PropertyType,True
8,PropertySubType,True
9,CountyOrParish,True


## 4. Convert date and numeric fields

In [11]:
# Convert date columns
date_cols = [
    "CloseDate",
    "PurchaseContractDate",
    "ListingContractDate",
    "ContractStatusChangeDate"
]

for col in date_cols:
    if col in sold.columns:
        sold[col] = pd.to_datetime(sold[col], errors="coerce")
        print(f"Converted to datetime: {col}")
    else:
        print(f"Missing date column: {col}")

# Convert numeric columns
numeric_cols = [
    "ClosePrice",
    "OriginalListPrice",
    "ListPrice",
    "LivingArea",
    "DaysOnMarket"
]

for col in numeric_cols:
    if col in sold.columns:
        sold[col] = pd.to_numeric(sold[col], errors="coerce")
        print(f"Converted to numeric: {col}")
    else:
        print(f"Missing numeric column: {col}")

Converted to datetime: CloseDate
Converted to datetime: PurchaseContractDate
Converted to datetime: ListingContractDate
Converted to datetime: ContractStatusChangeDate
Converted to numeric: ClosePrice
Converted to numeric: OriginalListPrice
Converted to numeric: ListPrice
Converted to numeric: LivingArea
Converted to numeric: DaysOnMarket


## 5. Create helper function for safe division

This avoids invalid ratios when the denominator is missing or zero.

In [12]:
def safe_divide(numerator, denominator):
    """
    Return numerator / denominator.
    If denominator is missing or zero, return NaN.
    """
    return np.where(
        denominator.notna() & (denominator != 0),
        numerator / denominator,
        np.nan
    )

## 6. Engineer Week 6 market metrics

Required metrics:

| Metric | Formula |
|---|---|
| Price Ratio | ClosePrice / OriginalListPrice |
| Price Per Sq Ft | ClosePrice / LivingArea |
| Days on Market | DaysOnMarket |
| Year / Month / YrMo | Derived from CloseDate |
| Close-to-Original-List Ratio | ClosePrice / OriginalListPrice |
| Listing-to-Contract Days | PurchaseContractDate - ListingContractDate |
| Contract-to-Close Days | CloseDate - PurchaseContractDate |


In [13]:
# 1. Price Ratio = ClosePrice / OriginalListPrice
if {"ClosePrice", "OriginalListPrice"}.issubset(sold.columns):
    sold["price_ratio"] = safe_divide(sold["ClosePrice"], sold["OriginalListPrice"])
else:
    sold["price_ratio"] = np.nan

# 2. Close-to-Original-List Ratio
# Same formula, clearer Tableau/business name
if {"ClosePrice", "OriginalListPrice"}.issubset(sold.columns):
    sold["close_to_original_list_ratio"] = safe_divide(sold["ClosePrice"], sold["OriginalListPrice"])
else:
    sold["close_to_original_list_ratio"] = np.nan

# 3. Price Per Sq Ft = ClosePrice / LivingArea
if {"ClosePrice", "LivingArea"}.issubset(sold.columns):
    sold["price_per_sqft"] = safe_divide(sold["ClosePrice"], sold["LivingArea"])
else:
    sold["price_per_sqft"] = np.nan

# 4. Days on Market
if "DaysOnMarket" in sold.columns:
    sold["days_on_market"] = sold["DaysOnMarket"]
else:
    sold["days_on_market"] = np.nan

# 5. Year / Month / YrMo from CloseDate
if "CloseDate" in sold.columns:
    sold["close_year"] = sold["CloseDate"].dt.year
    sold["close_month"] = sold["CloseDate"].dt.month
    sold["yrmo"] = sold["CloseDate"].dt.to_period("M").astype(str)
else:
    sold["close_year"] = np.nan
    sold["close_month"] = np.nan
    sold["yrmo"] = np.nan

# 6. Listing-to-Contract Days = PurchaseContractDate - ListingContractDate
if {"PurchaseContractDate", "ListingContractDate"}.issubset(sold.columns):
    sold["listing_to_contract_days"] = (
        sold["PurchaseContractDate"] - sold["ListingContractDate"]
    ).dt.days
else:
    sold["listing_to_contract_days"] = np.nan

# 7. Contract-to-Close Days = CloseDate - PurchaseContractDate
if {"CloseDate", "PurchaseContractDate"}.issubset(sold.columns):
    sold["contract_to_close_days"] = (
        sold["CloseDate"] - sold["PurchaseContractDate"]
    ).dt.days
else:
    sold["contract_to_close_days"] = np.nan

engineered_cols = [
    "price_ratio",
    "close_to_original_list_ratio",
    "price_per_sqft",
    "days_on_market",
    "close_year",
    "close_month",
    "yrmo",
    "listing_to_contract_days",
    "contract_to_close_days"
]

sold[engineered_cols].head()

,price_ratio,close_to_original_list_ratio,price_per_sqft,days_on_market,close_year,close_month,yrmo,listing_to_contract_days,contract_to_close_days
0,0.480962,0.480962,210.526316,777,2024,1,2024-01,777.0,65.0
1,NaN,NaN,NaN,901,2024,1,2024-01,901.0,0.0
2,0.600000,0.600000,NaN,865,2024,1,2024-01,887.0,25.0
3,0.711055,0.711055,NaN,830,2024,1,2024-01,832.0,76.0
4,0.769231,0.769231,NaN,805,2024,1,2024-01,809.0,145.0


## 7. Validate engineered columns

Check whether the new metrics are populated correctly and whether any columns have many missing values.


In [14]:
engineered_null_summary = (
    sold[engineered_cols]
    .isna()
    .sum()
    .reset_index()
)

engineered_null_summary.columns = ["engineered_column", "null_count"]
engineered_null_summary["null_pct"] = engineered_null_summary["null_count"] / len(sold)

engineered_null_summary

,engineered_column,null_count,null_pct
0,price_ratio,1733,0.002929
1,close_to_original_list_ratio,1733,0.002929
2,price_per_sqft,42273,0.071439
3,days_on_market,0,0.000000
4,close_year,0,0.000000
5,close_month,0,0.000000
6,yrmo,0,0.000000
7,listing_to_contract_days,11911,0.020129
8,contract_to_close_days,11831,0.019994


In [15]:
metric_summary_cols = [
    "price_ratio",
    "close_to_original_list_ratio",
    "price_per_sqft",
    "days_on_market",
    "listing_to_contract_days",
    "contract_to_close_days"
]

metric_summary = (
    sold[metric_summary_cols]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95, 0.99])
    .T
)

metric_summary

,count,mean,std,min,25%,50%,75%,90%,95%,99%,max
price_ratio,590000.0,46.464915,13254.663299,0.0,0.945234,0.998491,1.006897,1.053741,1.100011,1.274131,9.118000e+06
close_to_original_list_ratio,590000.0,46.464915,13254.663299,0.0,0.945234,0.998491,1.006897,1.053741,1.100011,1.274131,9.118000e+06
price_per_sqft,549460.0,476.579029,4337.516090,0.0,89.285714,415.335463,647.987820,897.255936,1110.038610,1744.747186,1.164067e+06
days_on_market,591733.0,43.292994,69.841987,-288.0,9.000000,22.000000,54.000000,103.000000,147.000000,290.000000,1.243000e+04
listing_to_contract_days,579822.0,50.560267,95.844601,-36407.0,11.000000,27.000000,62.000000,117.000000,166.000000,338.000000,1.465700e+04
contract_to_close_days,579902.0,26.590417,57.458615,-337.0,11.000000,24.000000,34.000000,49.000000,63.000000,119.000000,3.662900e+04


## 8. Create sample output table

This table shows the required new columns populated correctly.


In [16]:
sample_cols = [
    "CloseDate",
    "ClosePrice",
    "OriginalListPrice",
    "LivingArea",
    "DaysOnMarket",
    "price_ratio",
    "close_to_original_list_ratio",
    "price_per_sqft",
    "close_year",
    "close_month",
    "yrmo",
    "ListingContractDate",
    "PurchaseContractDate",
    "listing_to_contract_days",
    "contract_to_close_days",
    "PropertyType",
    "PropertySubType",
    "CountyOrParish",
    "MLSAreaMajor"
]

sample_cols_existing = [col for col in sample_cols if col in sold.columns]

week6_sample_output = sold[sample_cols_existing].head(20)

week6_sample_output

,CloseDate,ClosePrice,OriginalListPrice,LivingArea,DaysOnMarket,price_ratio,close_to_original_list_ratio,price_per_sqft,close_year,close_month,yrmo,ListingContractDate,PurchaseContractDate,listing_to_contract_days,contract_to_close_days,PropertyType,PropertySubType,CountyOrParish,MLSAreaMajor
0,2024-01-26,240000.0,499000.0,1140.0,777,0.480962,0.480962,210.526316,2024,1,2024-01,2021-10-06,2023-11-22,777.0,65.0,Residential,Condominium,San Mateo,699 - Not Defined
1,2024-01-24,950.0,0.0,NaN,901,NaN,NaN,NaN,2024,1,2024-01,2021-08-06,2024-01-24,901.0,0.0,CommercialLease,Retail,San Bernardino,NaN
2,2024-01-16,45000.0,75000.0,NaN,865,0.600000,0.600000,NaN,2024,1,2024-01,2021-07-18,2023-12-22,887.0,25.0,Land,NaN,Kern,LKIS - Lake Isabella
3,2024-01-08,141500.0,199000.0,NaN,830,0.711055,0.711055,NaN,2024,1,2024-01,2021-07-14,2023-10-24,832.0,76.0,Land,NaN,San Bernardino,NaN
4,2024-01-17,15000.0,19500.0,NaN,805,0.769231,0.769231,NaN,2024,1,2024-01,2021-06-07,2023-08-25,809.0,145.0,Land,NaN,Los Angeles,LLO - Llano
5,2024-01-05,815000.0,759900.0,1974.0,33,1.072510,1.072510,412.867275,2024,1,2024-01,2021-03-08,2021-06-30,114.0,919.0,Residential,SingleFamilyResidence,San Diego,91950 - National City
6,2024-01-17,4500.0,4200.0,2834.0,33,1.071429,1.071429,1.587862,2024,1,2024-01,2021-05-06,2024-01-17,986.0,0.0,ResidentialLease,SingleFamilyResidence,San Diego,91913 - Chula Vista
7,2024-01-29,1200.0,5500.0,2462.0,1004,0.218182,0.218182,0.487409,2024,1,2024-01,2021-04-30,2024-01-29,1004.0,0.0,ResidentialLease,SingleFamilyResidence,Orange,N9 - Lower Newport Bay - Balboa Island
8,2024-01-03,3999.0,3990.0,NaN,140,1.002256,1.002256,NaN,2024,1,2024-01,2021-04-22,2021-09-09,140.0,846.0,Land,NaN,Modoc,ALT - Alturas-Modoc
9,2024-01-05,810000.0,739900.0,1974.0,228,1.094743,1.094743,410.334347,2024,1,2024-01,2021-03-08,2021-11-18,255.0,778.0,Residential,SingleFamilyResidence,San Diego,91950 - National City


## 9. Segmented summary table: County

The handbook requires at least one segmented summary table grouped by `PropertyType` or `CountyOrParish`.  
This section creates a county-level market summary.


In [17]:
if "CountyOrParish" in sold.columns:
    county_summary = (
        sold.groupby("CountyOrParish", dropna=False)
        .agg(
            closed_sales=("ClosePrice", "count"),
            median_close_price=("ClosePrice", "median"),
            average_close_price=("ClosePrice", "mean"),
            median_price_per_sqft=("price_per_sqft", "median"),
            average_days_on_market=("days_on_market", "mean"),
            median_days_on_market=("days_on_market", "median"),
            average_close_to_original_list_ratio=("close_to_original_list_ratio", "mean"),
            median_listing_to_contract_days=("listing_to_contract_days", "median"),
            median_contract_to_close_days=("contract_to_close_days", "median")
        )
        .reset_index()
        .sort_values("closed_sales", ascending=False)
    )
else:
    county_summary = pd.DataFrame()
    print("CountyOrParish column not found.")

county_summary.head(20)

,CountyOrParish,closed_sales,median_close_price,average_close_price,median_price_per_sqft,average_days_on_market,median_days_on_market,average_close_to_original_list_ratio,median_listing_to_contract_days,median_contract_to_close_days
20,Los Angeles,179270,630000.0,8.343656e+05,440.918742,42.735228,24.0,66.959492,28.0,23.0
31,Orange,79211,670000.0,9.183483e+05,518.363552,34.721195,18.0,47.387137,26.0,20.0
37,Riverside,74763,525000.0,5.604430e+05,291.519435,52.404245,31.0,100.641208,39.0,27.0
41,San Diego,62813,805000.0,1.204094e+06,539.044289,33.979431,18.0,2.152270,24.0,24.0
40,San Bernardino,54386,430000.0,4.442400e+05,288.229854,55.912147,27.5,45.251376,34.0,29.0
0,Alameda,21764,1050000.0,1.267594e+06,683.378653,30.276006,14.0,47.043796,14.0,23.0
7,Contra Costa,20957,755000.0,1.043253e+06,492.511521,32.263301,16.0,1.192933,16.0,22.0
47,Santa Clara,20872,1490000.0,1.740153e+06,907.414960,25.378354,11.0,1.034692,11.0,24.0
60,Ventura,18140,735000.0,7.768499e+05,456.674473,43.796141,28.0,1.502416,33.0,18.0
45,San Mateo,9038,1450000.0,1.809089e+06,937.500000,33.646161,13.0,2.390754,13.0,21.0


## 10. Additional segmented summary: Property type and subtype

This supports Tableau filtering and later market comparison.


In [19]:
if "MLSAreaMajor" in sold.columns:
    mls_area_summary = (
        sold.groupby("MLSAreaMajor", dropna=False)
        .agg(
            closed_sales=("ClosePrice", "count"),
            median_close_price=("ClosePrice", "median"),
            average_close_price=("ClosePrice", "mean"),
            median_price_per_sqft=("price_per_sqft", "median"),
            average_days_on_market=("days_on_market", "mean"),
            average_close_to_original_list_ratio=("close_to_original_list_ratio", "mean")
        )
        .reset_index()
        .sort_values("closed_sales", ascending=False)
    )
else:
    mls_area_summary = pd.DataFrame()
    print("MLSAreaMajor column not found.")

mls_area_summary.head(20)

,MLSAreaMajor,closed_sales,median_close_price,average_close_price,median_price_per_sqft,average_days_on_market,average_close_to_original_list_ratio
1126,NaN,65077,700000.0,9.487697e+05,483.205657,41.285392,22.611812
307,699 - Not Defined,54376,975000.0,1.330261e+06,683.760684,37.606021,1.194103
993,SRCAR - Southwest Riverside County,25762,522105.0,5.062073e+05,274.175429,48.764692,57.701244
137,252 - Riverside,6782,614950.0,5.732254e+05,346.971520,42.785462,1.008491
132,248 - Corona,4373,695000.0,6.114670e+05,364.409970,39.603019,236.163850
299,686 - Ontario,3871,571613.0,4.787119e+05,372.549020,37.700077,1.420904
700,LAC - Lancaster,3842,442250.0,3.987401e+05,262.054507,58.985685,1.488052
150,274 - San Bernardino,3809,460000.0,4.285527e+05,326.970357,44.031504,1.952427
1076,VIC - Victorville,3804,408000.0,4.653554e+05,221.945411,50.779706,18.999620
301,688 - Rancho Cucamonga,3671,645000.0,5.832469e+05,394.907253,34.193680,1.270014


## 12. Additional segmented summary: Listing office and buyer office

This supports later competitive intelligence analysis.


In [20]:
office_group_cols = []

if "ListOfficeName" in sold.columns:
    office_group_cols.append("ListOfficeName")

if "BuyerOfficeName" in sold.columns:
    office_group_cols.append("BuyerOfficeName")

if office_group_cols:
    office_summary = (
        sold.groupby(office_group_cols, dropna=False)
        .agg(
            closed_sales=("ClosePrice", "count"),
            total_sales_volume=("ClosePrice", "sum"),
            median_close_price=("ClosePrice", "median"),
            average_days_on_market=("days_on_market", "mean"),
            average_close_to_original_list_ratio=("close_to_original_list_ratio", "mean")
        )
        .reset_index()
        .sort_values("total_sales_volume", ascending=False)
    )
else:
    office_summary = pd.DataFrame()
    print("ListOfficeName / BuyerOfficeName columns not found.")

office_summary.head(20)

,ListOfficeName,BuyerOfficeName,closed_sales,total_sales_volume,median_close_price,average_days_on_market,average_close_to_original_list_ratio
57258,Compass,Compass,13384,1.605740e+10,715000.0,35.973476,1.362568
51438,Coldwell Banker Realty,Coldwell Banker Realty,8366,7.643825e+09,9500.0,39.863615,1.659091
51465,Coldwell Banker Realty,Compass,1789,3.137096e+09,1377000.0,34.846842,1.562263
57224,Compass,Coldwell Banker Realty,1491,2.897435e+09,1468000.0,33.638498,1.586192
55231,Coldwell Banker West,Coldwell Banker West,724,2.185868e+09,645000.0,28.563536,9.223620
120198,Keller Williams Realty,Keller Williams Realty,2653,1.649093e+09,492000.0,35.034678,20.768895
19603,Berkshire Hathaway HomeServices California Pro...,Berkshire Hathaway HomeServices California Pro...,2204,1.627889e+09,8225.0,45.785844,1.295573
60786,Compass,NaN,890,1.548547e+09,1399000.0,29.446067,1.030147
103292,Intero Real Estate Services,Intero Real Estate Services,1019,1.492717e+09,1295000.0,26.819431,1.021561
83791,First Team Real Estate,First Team Real Estate,2522,1.412654e+09,8500.0,33.155432,1.371663


## Part B: Listing data metrics

In [24]:
listing_required_cols = [
    "ListingContractDate",
    "ListPrice",
    "LivingArea",
    "DaysOnMarket",
    "PropertyType",
    "PropertySubType",
    "CountyOrParish",
    "MLSAreaMajor",
    "ListOfficeName",
    "City",
    "PostalCode"
]

listing_column_check = pd.DataFrame({
    "column": listing_required_cols,
    "exists": [col in listings.columns for col in listing_required_cols]
})

listing_column_check

,column,exists
0,ListingContractDate,True
1,ListPrice,True
2,LivingArea,True
3,DaysOnMarket,True
4,PropertyType,True
5,PropertySubType,True
6,CountyOrParish,True
7,MLSAreaMajor,True
8,ListOfficeName,True
9,City,True


## Convert Listing Date and Numeric Fields

The listing-side metrics require ListingContractDate, ListPrice, LivingArea, and DaysOnMarket to be in the correct data types.

In [25]:
# Convert listing date field
if "ListingContractDate" in listings.columns:
    listings["ListingContractDate"] = pd.to_datetime(
        listings["ListingContractDate"],
        errors="coerce"
    )

# Convert listing numeric fields
listing_numeric_cols = [
    "ListPrice",
    "LivingArea",
    "DaysOnMarket"
]

for col in listing_numeric_cols:
    if col in listings.columns:
        listings[col] = pd.to_numeric(listings[col], errors="coerce")
        print(f"Converted {col} to numeric")
    else:
        print(f"Missing column: {col}")

Converted ListPrice to numeric
Converted LivingArea to numeric
Converted DaysOnMarket to numeric


## Create Listing-Side Market Metrics

This section creates listing-side features for market activity analysis, including new listing time variables and list price per square foot.

In [26]:
# Year / Month / YrMo from ListingContractDate
listings["list_year"] = listings["ListingContractDate"].dt.year
listings["list_month"] = listings["ListingContractDate"].dt.month
listings["list_yrmo"] = listings["ListingContractDate"].dt.to_period("M").astype(str)

# List Price Per Sq Ft = ListPrice / LivingArea
listings["list_price_per_sqft"] = safe_divide(
    listings["ListPrice"],
    listings["LivingArea"]
)

# Keep listing Days on Market
if "DaysOnMarket" in listings.columns:
    listings["listing_days_on_market"] = listings["DaysOnMarket"]
else:
    listings["listing_days_on_market"] = np.nan

listing_engineered_cols = [
    "list_year",
    "list_month",
    "list_yrmo",
    "list_price_per_sqft",
    "listing_days_on_market"
]

listings[listing_engineered_cols].head()

,list_year,list_month,list_yrmo,list_price_per_sqft,listing_days_on_market
0,2024,1,2024-01,93.750000,0
1,2024,1,2024-01,NaN,0
2,2024,1,2024-01,1029.976941,127
3,2024,1,2024-01,896.700143,1
4,2024,1,2024-01,969.230769,1


## Create monthly new listings summary

In [27]:
monthly_new_listings = (
    listings.groupby("list_yrmo", dropna=False)
    .agg(
        new_listings=("ListingContractDate", "count"),
        median_list_price=("ListPrice", "median"),
        average_list_price=("ListPrice", "mean"),
        median_list_price_per_sqft=("list_price_per_sqft", "median"),
        average_listing_days_on_market=("listing_days_on_market", "mean")
    )
    .reset_index()
    .sort_values("list_yrmo")
)

monthly_new_listings.head(20)

,list_yrmo,new_listings,median_list_price,average_list_price,median_list_price_per_sqft,average_listing_days_on_market
0,2024-01,27454,599000.0,9.358645e+05,409.865029,33.073905
1,2024-02,27447,625000.0,9.596700e+05,431.324531,26.749590
2,2024-03,32282,635141.5,1.000743e+06,432.593395,20.041045
3,2024-04,36503,688065.0,1.047444e+06,460.084175,10.082295
4,2024-05,38796,689000.0,1.061820e+06,459.718670,10.392231
5,2024-06,35893,669147.5,1.013638e+06,447.880299,9.582203
6,2024-07,36340,647500.0,9.581458e+05,437.043054,9.230930
7,2024-08,35305,632850.0,9.626102e+05,427.830596,7.944569
8,2024-09,34625,650000.0,1.022890e+06,441.262855,10.809877
9,2024-10,34730,625000.0,9.529126e+05,424.126221,11.392226


## Create listing summary by county

In [28]:
listing_property_summary = (
    listings.groupby(["PropertyType", "PropertySubType"], dropna=False)
    .agg(
        new_listings=("ListingContractDate", "count"),
        median_list_price=("ListPrice", "median"),
        average_list_price=("ListPrice", "mean"),
        median_list_price_per_sqft=("list_price_per_sqft", "median"),
        average_listing_days_on_market=("listing_days_on_market", "mean")
    )
    .reset_index()
    .sort_values("new_listings", ascending=False)
)

listing_property_summary.head(20)

,PropertyType,PropertySubType,new_listings,median_list_price,average_list_price,median_list_price_per_sqft,average_listing_days_on_market
54,Residential,SingleFamilyResidence,394000,924900.0,1.489540e+06,540.010926,18.979122
43,Residential,Condominium,97715,640000.0,8.117461e+05,582.772544,21.584230
90,ResidentialLease,SingleFamilyResidence,82876,4750.0,8.710639e+03,2.824087,18.106231
36,Land,NaN,55954,110000.0,5.841985e+05,NaN,19.350449
79,ResidentialLease,Condominium,49101,3500.0,4.484020e+03,3.118750,19.579092
58,Residential,Townhouse,31419,819000.0,9.641405e+05,570.108696,19.493077
38,ManufacturedInPark,NaN,23698,184999.0,2.207130e+05,156.250000,24.792514
76,ResidentialLease,Apartment,16559,2625.0,3.820885e+03,3.188421,20.129356
94,ResidentialLease,Townhouse,12998,3950.0,5.321901e+03,2.690972,17.926296
75,ResidentialIncome,NaN,11010,1649000.0,2.238729e+06,455.248813,20.966667
